<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/12_new_test_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q jiwer

In [2]:
import csv
import re
from pathlib import Path

import jiwer
import pandas as pd
import soundfile as sf
import torch
from google.colab import drive
from transformers import Wav2Vec2ForCTC, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

using: cuda
Mounted at /content/drive


## Download the data
Same as Day 5.

In [3]:
work_folder = Path("/content/openslr54")
work_folder.mkdir(exist_ok=True)

base_url = "https://openslr.trmal.net/resources/54"
!wget -q -c -P {work_folder} {base_url}/utt_spk_text.tsv
!wget -q -c -P {work_folder} {base_url}/asr_nepali_0.zip
!unzip -q -n {work_folder}/asr_nepali_0.zip -d {work_folder}/audio_zip_0

In [4]:
all_rows = pd.read_csv(
    work_folder / "utt_spk_text.tsv",
    sep="\t",
    header=None,
    names=["file_id", "speaker_id", "text"],
    dtype=str,
    keep_default_na=False,
    quoting=csv.QUOTE_NONE,
)

path_by_id = {}
for file_path in (work_folder / "audio_zip_0").rglob("*.flac"):
    path_by_id[file_path.stem] = file_path

all_rows["audio_path"] = all_rows["file_id"].map(path_by_id)
rows_with_audio = all_rows.dropna(subset=["audio_path"])
print(len(rows_with_audio), "rows with audio")

9935 rows with audio


## Pick 300 new files
Leave out every speaker who is in the first 200 files, so the test is really new.

In [5]:
old_ids = pd.read_csv(folder / "scores_200.csv", dtype={"file_id": str})["file_id"]
used_speakers = set(all_rows[all_rows["file_id"].isin(old_ids)]["speaker_id"])

new_pool = rows_with_audio[~rows_with_audio["speaker_id"].isin(used_speakers)]
print(len(used_speakers), "speakers used before |", len(new_pool), "rows in the new pool")

how_many = min(300, len(new_pool))
test_rows = new_pool.sample(n=how_many, random_state=11).reset_index(drop=True)
test_rows["audio_path"] = test_rows["audio_path"].astype(str)
print(len(test_rows), "new files chosen")

166 speakers used before | 6376 rows in the new pool
300 new files chosen


## Load the model and run it
Same as Day 5. Nepali is "npi".

In [6]:
model_name = "facebook/mms-1b-all"
processor = AutoProcessor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

processor.tokenizer.set_target_lang("npi")
model.load_adapter("npi")
model = model.to(device)
model.eval()
print("model ready")

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.npi.safetensors: reconstructing file:   0%|          |  0.00B / 9.28MB            

adapter.npi.safetensors: downloading bytes:           |  0.00B            

model ready


In [7]:
def get_prediction(audio_file):
    audio, sample_rate = sf.read(audio_file)
    model_input = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        scores = model(**model_input).logits
    best_ids = torch.argmax(scores, dim=-1)[0]
    return processor.decode(best_ids)


model_texts = []
for number, audio_path in enumerate(test_rows["audio_path"]):
    model_texts.append(get_prediction(audio_path))
    if (number + 1) % 50 == 0:
        print(number + 1, "done")

test_rows["model_text"] = model_texts

50 done
100 done
150 done
200 done
250 done
300 done


In [11]:
results = test_rows[["file_id", "speaker_id", "text", "model_text"]].rename(columns={"text": "correct_text"})
results["wer"] = [jiwer.wer(c, m) for c, m in zip(results["correct_text"], results["model_text"])]
results["cer"] = [jiwer.cer(c, m) for c, m in zip(results["correct_text"], results["model_text"])]

results.to_csv(folder / "predictions_test.csv", index=False)

print("WER:", round(jiwer.wer(list(results["correct_text"]), list(results["model_text"])) * 100, 1), "%")
print("CER:", round(jiwer.cer(list(results["correct_text"]), list(results["model_text"])) * 100, 1), "%")

WER: 59.8 %
CER: 19.8 %


## Frozen rules
Same sorter and folding as Days 9 and 11. Do not change them.

In [12]:
digit_map = str.maketrans("०१२३४५६७८९", "0123456789")
vowel_signs = "ािीुूृेैोौ्ंँः"


def remove_punctuation(text):
    for mark in "।,.?!;:\"'":
        text = text.replace(mark, " ")
    return " ".join(text.split())


def has_latin_letters(text):
    return any("a" <= letter.lower() <= "z" for letter in text)


def count_broken_words(text):
    count = 0
    for word in text.split():
        if word[0] in vowel_signs:
            count += 1
    return count


def find_numbers(text):
    return re.findall(r"[0-9]+", text.translate(digit_map))


def edit_distance(first, second):
    previous_row = list(range(len(second) + 1))
    for i, letter_1 in enumerate(first, start=1):
        current_row = [i]
        for j, letter_2 in enumerate(second, start=1):
            cost = 0 if letter_1 == letter_2 else 1
            current_row.append(min(previous_row[j] + 1, current_row[j - 1] + 1, previous_row[j - 1] + cost))
        previous_row = current_row
    return previous_row[-1]


def few_letters_differ(a, b):
    a_short = a.translate(digit_map).replace(" ", "")
    b_short = b.translate(digit_map).replace(" ", "")
    return edit_distance(a_short, b_short) <= 3


def find_error_type(correct, model):
    if correct == model:
        return "exact"

    a = remove_punctuation(correct)
    b = remove_punctuation(model)
    if a == b:
        return "punctuation_only"
    if a.replace(" ", "") == b.replace(" ", ""):
        return "spacing_only"
    if b == "":
        return "big_difference"

    if count_broken_words(model) >= 2:
        return "junk"
    if has_latin_letters(model):
        return "latin_mix"

    a_digits = a.translate(digit_map)
    b_digits = b.translate(digit_map)
    if a_digits.replace(" ", "") == b_digits.replace(" ", ""):
        return "digit_script_only"
    if find_numbers(a) != find_numbers(b):
        return "number_wrong"

    if few_letters_differ(a, b):
        return "small_difference"
    return "big_difference"


def fold_text(text):
    for mark in "।,.?!;:\"'":
        text = text.replace(mark, " ")
    text = text.translate(digit_map)
    text = text.replace("ङ्", "ं").replace("ँ", "ं")
    text = text.replace("ी", "ि").replace("ू", "ु")
    text = text.replace("श", "स").replace("ष", "स")
    text = text.replace("्", "")
    return text.replace(" ", "")

## Sort the new rows and score them with folded CER
Done now, before anybody rates, so we cannot be influenced by the ratings.

In [13]:
group_of = {
    "exact": "exact",
    "punctuation_only": "harmless",
    "spacing_only": "harmless",
    "digit_script_only": "harmless",
    "small_difference": "needs a human",
    "junk": "real mistake",
    "latin_mix": "real mistake",
    "number_wrong": "real mistake",
    "big_difference": "real mistake",
}

results["error_type"] = [find_error_type(c, m) for c, m in zip(results["correct_text"], results["model_text"])]
results["group"] = results["error_type"].map(group_of)
results["folded_cer"] = [jiwer.cer(fold_text(c), fold_text(m)) for c, m in zip(results["correct_text"], results["model_text"])]

print(results["error_type"].value_counts())
print()
print(results["group"].value_counts())

error_type
small_difference     119
exact                 73
latin_mix             59
big_difference        18
spacing_only          13
number_wrong          11
junk                   3
digit_script_only      3
punctuation_only       1
Name: count, dtype: int64

group
needs a human    119
real mistake      91
exact             73
harmless          17
Name: count, dtype: int64


## Rating sheet for the test set
Same recipe as round 2: all harmless and needs-a-human rows, plus 20 control rows. No scores or types in the sheet.

In [14]:
need_rating = results[results["group"].isin(["harmless", "needs a human"])]
control_rows = results[results["group"] == "real mistake"].sample(n=20, random_state=5)

rating_rows = pd.concat([need_rating, control_rows]).sample(frac=1, random_state=5).reset_index(drop=True)

rating_sheet_test = rating_rows[["file_id", "correct_text", "model_text"]]
rating_sheet_test.to_csv(folder / "rating_sheet_test.csv", index=False)
rating_rows.to_csv(folder / "rating_rows_private_test.csv", index=False)
results.to_csv(folder / "sorted_test.csv", index=False)

print(len(rating_sheet_test), "rows to rate")
print(rating_rows["group"].value_counts())

156 rows to rate
group
needs a human    119
real mistake      20
harmless          17
Name: count, dtype: int64


## Notes
- WER and CER on the new files vs the first 200:
- how many rows to rate: